In [2]:
#setup

#conda create --name sds-project python=3.12
#conda install --channel conda-forge pandas numpy matplotlib requests geopandas


import pandas as pd
import requests
import geopandas as gpd

In [3]:
MAP_KEY = '5eae605403f5deded880b550afef3667'

def get_transaction_count() :
  count = 0
  try:
    response = requests.get(url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

In [4]:
#sensors:
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
df = pd.read_csv(da_url)
display(df)

#set one sensor
sensor = df["data_id"][2] #VIIRS has better resolution than other sensors (375m vs 1000m), NRT means only a few minutes lag
print("Current sensor name: ", sensor)

,data_id,min_date,max_date
0,MODIS_NRT,2026-02-01,2026-05-08
1,MODIS_SP,2000-11-01,2026-01-31
2,VIIRS_NOAA20_NRT,2026-03-01,2026-05-08
3,VIIRS_NOAA20_SP,2018-04-01,2026-02-28
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-08
5,VIIRS_SNPP_NRT,2026-03-01,2026-05-08
6,VIIRS_SNPP_SP,2012-01-20,2026-02-28
7,LANDSAT_NRT,2022-06-20,2026-05-07
8,GOES_NRT,2022-08-09,2026-05-08
9,BA_MODIS,2000-11-01,2026-02-01


Current sensor name:  VIIRS_NOAA20_NRT


In [9]:
date

In [10]:
#retrieve data 

area = "world"
day_range = 1
date = None
area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + f'/{sensor}/{area}/{day_range}'
start_count = get_transaction_count()
df_area = pd.read_csv(area_url)
end_count = get_transaction_count()
print ('We used %i transactions.' % (end_count-start_count))

df_area

We used 36 transactions.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,-1.52327,29.24781,318.93,0.6,0.53,2026-05-08,1,N20,VIIRS,n,2.0NRT,275.52,5.14,N
1,-1.52171,29.24591,334.69,0.6,0.53,2026-05-08,1,N20,VIIRS,n,2.0NRT,275.26,7.18,N
2,-1.40824,29.23271,299.12,0.6,0.53,2026-05-08,1,N20,VIIRS,n,2.0NRT,264.49,1.63,N
3,-1.40771,29.23770,295.15,0.6,0.53,2026-05-08,1,N20,VIIRS,n,2.0NRT,264.71,1.31,N
4,-1.40452,29.24039,297.20,0.6,0.53,2026-05-08,1,N20,VIIRS,n,2.0NRT,265.89,1.13,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11064,42.54409,-122.40182,299.98,0.4,0.40,2026-05-08,957,N20,VIIRS,n,2.1URT,282.57,0.79,N
11065,43.17841,-115.73341,309.35,0.4,0.40,2026-05-08,957,N20,VIIRS,n,2.1URT,278.58,1.03,N
11066,43.39116,-121.61675,303.50,0.4,0.40,2026-05-08,957,N20,VIIRS,n,2.1URT,275.49,1.34,N
11067,43.78008,-120.92174,301.76,0.4,0.40,2026-05-08,957,N20,VIIRS,n,2.1URT,266.71,2.04,N


In [12]:
area_gpd = gpd.GeoDataFrame(
    df_area, geometry=gpd.points_from_xy(df_area["longitude"], df_area["latitude"], crs = 4326)
)
area_gpd.explore(column = "bright_ti4", cmap = "YlOrRd")